# KAN Autoencoder (KAN-AE)

_for spatial behavior representation through encoding_

In [ ]:
import pandas as pd
import numpy as np
import os

import keras
import torch
import torch.nn as nn
from kan import KANLayer
from tqdm import tqdm
from torch.utils.data import TensorDataset, DataLoader

In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"]="5,6" # no need to write "0"

## Loading Data

In [ ]:
BASE_PATH = "<path to experiment directory>"
MODEL_DATA_PATH = f"{BASE_PATH}/<path the genuine AV/DT data split>"

MODEL_PATH = f"{BASE_PATH}/<path to export KAN-AE model>"
ENC_DATA_PATH = f"{BASE_PATH}/<path to export KAN-AE encoded data>"

In [ ]:
X_train = np.load(f"{MODEL_DATA_PATH}/X_train.npy")
X_test = np.load(f"{MODEL_DATA_PATH}/X_test.npy")
X_val = np.load(f"{MODEL_DATA_PATH}/X_val.npy")

y_train = np.load(f"{MODEL_DATA_PATH}/y_train_.npy")
y_test = np.load(f"{MODEL_DATA_PATH}/y_test.npy")
y_val = np.load(f"{MODEL_DATA_PATH}/y_val.npy")

In [ ]:
# Check shapes
print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

# Expected:   (N, seq, features), (N,)

---

In [ ]:
# seperate individual timestemp data (spatial signals)
# it requires removing the redundend spatial data comming form overlapping windows.
def convertTo2D(X, y):
    """
    Convert 3D time-series data to 2D by treating each timestamp
    as an independent sample, and expand y accordingly.

    Parameters
    ----------
    X : np.ndarray
        Shape: (n_samples, n_timestamps, n_features)

    y : np.ndarray
        Shape: (n_samples,)

    Returns
    -------
    X_2d : np.ndarray
        Shape: (n_samples * n_timestamps, n_features)

    y_2d : np.ndarray
        Shape: (n_samples * n_timestamps,)
    """

    n_samples, n_timestamps, n_features = X.shape

    # Reshape X
    X_2d = X.reshape(n_samples * n_timestamps, n_features)

    # Repeat each label for all timestamps
    y_2d = np.repeat(y, n_timestamps)

    return X_2d, y_2d

In [ ]:
# converting to the 2D from 3D
X_train, y_train = convertTo2D(X_train.copy(), y_train.copy())
X_val, y_val = convertTo2D(X_val.copy(), y_val.copy())
X_test, y_test = convertTo2D(X_test.copy(), y_test.copy())

In [ ]:
# remove duplicate rows from X_2d
X_train_2D, unique_idx = np.unique(X_train, axis=0, return_index=True)
X_val_2D, unique_idx = np.unique(X_val, axis=0, return_index=True)
X_test_2D, unique_idx = np.unique(X_test, axis=0, return_index=True)

In [ ]:
# Check shapes
print(X_train_2D.shape, y_train.shape)
print(X_val_2D.shape, y_val.shape)
print(X_test_2D.shape, y_test.shape)

<font color="red"> **y will not be required for autoencoder** </font>

---

## Model Building

In [ ]:
CLASS_COUNTS = len(set(y_train))
CLASS_COUNTS

In [ ]:
FEATURES = X_train_2D.shape[1]
FEATURES

### Kolmogorov-Arnold Net's Autoencoder (KAN-AE)

In [ ]:
# ──────────────────────────────────────────────
# Row-wise KAN layer → applies KANLayer on each of the rows
# ──────────────────────────────────────────────
class KANRowLayer(nn.Module):
    def __init__(self, in_dim, out_dim, num=5, k=3, device='cpu'):
        super().__init__()
        self.kan = KANLayer(
            in_dim=in_dim, out_dim=out_dim, num=num, k=k,
            device=device, save_plot_data=False
        )

    def forward(self, x):
        x = self.kan(x)
        if isinstance(x, tuple):
            x = x[0]  # extract tensor, ignore metadata
        return x

In [ ]:
# ──────────────────────────────────────────────
# Encoder
# ──────────────────────────────────────────────
class Encoder(nn.Module):
    def __init__(self, latent_dim=128, device='cpu'):
        super().__init__()
        self.block1 = KANRowLayer(FEATURES, 32, device=device)
        self.block2 = KANRowLayer(32, 64, device=device)
        self.block3 = KANRowLayer(64, latent_dim, device=device)

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        z = self.block3(x)
        return z

In [ ]:
# ──────────────────────────────────────────────
# Decoder (symmetric encoder)
# ──────────────────────────────────────────────
class Decoder(nn.Module):
    def __init__(self, latent_dim=128, device='cpu'):
        super().__init__()
        self.block1 = KANRowLayer(latent_dim, 64, device=device)
        self.block2 = KANRowLayer(64, 32, device=device)
        self.block3 = KANRowLayer(32, FEATURES, device=device)

    def forward(self, z):
        x = self.block1(z)
        x = self.block2(x)
        x = self.block3(x)
        return x

In [ ]:
# ──────────────────────────────────────────────
# Full Autoencoder
# ──────────────────────────────────────────────
class KANAutoencoder(nn.Module):
    def __init__(self, latent_dim=128, device='cpu'):
        super().__init__()
        self.encoder = Encoder(latent_dim, device)
        self.decoder = Decoder(latent_dim, device)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

---

### Data Prep.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train_t = torch.tensor(X_train_2D, dtype=torch.float32).to(device)
X_val_t   = torch.tensor(X_val_2D,   dtype=torch.float32).to(device)
X_test_t  = torch.tensor(X_test_2D,  dtype=torch.float32).to(device)

In [ ]:
# data loaders for tensor
batch_size = 64 # tuned

train_loader = DataLoader(TensorDataset(X_train_t, X_train_t),
                          batch_size=batch_size, shuffle=True)

val_loader = DataLoader(TensorDataset(X_val_t, X_val_t),
                        batch_size=batch_size, shuffle=False)

test_loader = DataLoader(TensorDataset(X_test_t, X_test_t),
                         batch_size=batch_size, shuffle=False)

---

### Training 

In [ ]:
autoencoder = KANAutoencoder(latent_dim=128, device=device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3)

num_epochs = 15
best_val_loss = float("inf")
patience = 5
wait = 0

In [ ]:
# autoencoder.summary()
def model_summary(model):
    print(model)
    print("Total trainable parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

# custom summary
model_summary(autoencoder)

#### With Earlystop

In [ ]:
for epoch in range(num_epochs):
    autoencoder.train()
    train_loss = 0.0

    for batch_x, _ in train_loader:
        optimizer.zero_grad()
        x_hat = autoencoder(batch_x)
        loss = criterion(x_hat, batch_x)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)

    # Validation
    autoencoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for batch_x, _ in val_loader:
            x_hat = autoencoder(batch_x)
            val_loss += criterion(x_hat, batch_x).item()
    val_loss /= len(val_loader)

    print(f"Epoch {epoch+1}/{num_epochs} - Train: {train_loss:.4f}  Val: {val_loss:.4f}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        wait = 0
        torch.save(autoencoder.state_dict(), f"{MODEL_PATH}/best_kanae.pth")
    else:
        wait += 1
        if wait >= patience:
            print("Early stopping triggered.")
            break

### Evaluation

In [ ]:
# loading encoder
best_model = KANAutoencoder(latent_dim=128, device=device)#.to(device)
best_model.load_state_dict(torch.load(f"{MODEL_PATH}/best_kanae.pth"))
best_model.eval()

In [ ]:
## recons. error
criterion = nn.MSELoss()

test_loss = 0.0
with torch.no_grad():
    for batch_x, _ in test_loader:
        reconstructed = best_model(batch_x)
        loss = criterion(reconstructed, batch_x)
        test_loss += loss.item()

test_loss /= len(test_loader)
print(f"\nFINAL TEST LOSS (Reconstruction MSE): {test_loss:.6f}")

<font color="red"> **On env's eval data:** 0.003950; 0.000674 (50 AVs)  </font>

In [ ]:
# sample wise error/loss
per_sample_errors = []

with torch.no_grad():
    for batch_x, _ in test_loader:
        reconstructed = best_model(batch_x)
        # print(reconstructed.shape)
        mse = torch.mean((reconstructed - batch_x)**2, dim=1)
        per_sample_errors.extend(mse.cpu().numpy())

per_sample_errors = np.array(per_sample_errors)

print("Sample error stats:")
print("  Mean:", per_sample_errors.mean())
print("  Std :", per_sample_errors.std())
print("  Max :", per_sample_errors.max())

## Extract Encodings

### Recall Original Data

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
## keeping tensor on CPU
X_train_t = torch.tensor(X_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val, dtype=torch.float32)
X_test_t  = torch.tensor(X_test, dtype=torch.float32)

In [ ]:
# data loaders for tensor
batch_size = 64 # tuned

train_loader = DataLoader(
    TensorDataset(X_train_t),
    batch_size=batch_size,
    shuffle=False
)

val_loader = DataLoader(
    TensorDataset(X_val_t),
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    TensorDataset(X_test_t),
    batch_size=batch_size,
    shuffle=False
)

### Encode Overlaped Data

#### Loading Model

In [ ]:
# getting model on GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

autoencoder = KANAutoencoder(latent_dim=128, device=device).to(device)
autoencoder.load_state_dict(
    torch.load(f"{MODEL_PATH}/best_kanae.pth")
)
autoencoder.eval()

#### Encoding

In [ ]:
# preventing entire data push to CUDA
def encode_dataset(model, loader, device):
    latent_batches = []
    model.eval()

    with torch.no_grad():
        for (x_batch,) in loader:
            # move ONLY batch to GPU
            x_batch = x_batch.to(device)
            # encode
            z = model.encoder(x_batch)
            # move back to CPU immediately
            latent_batches.append(z.cpu())
            # cleanup
            del x_batch, z

    return torch.cat(latent_batches, dim=0).numpy()

In [ ]:
Z_train = encode_dataset(autoencoder, train_loader, device)
Z_val   = encode_dataset(autoencoder, val_loader, device)
Z_test  = encode_dataset(autoencoder, test_loader, device)

In [ ]:
Z_train.shape, Z_val.shape, Z_test.shape, y_train.shape, y_val.shape, y_test.shape

<font color="red"> **fit PCA on train data and reduce dimentions of all 3 encoded datasets** </font>

---

## Export for Bi-GRU

In [ ]:
# np.save(f'{MODEL_DATA_PATH}/X_train_pca_enc.npy', PCA_train)
# np.save(f'{MODEL_DATA_PATH}/X_val_pca_enc.npy', PCA_val)
# np.save(f'{MODEL_DATA_PATH}/X_test_pca_enc.npy', PCA_test)

# np.save(f'{MODEL_DATA_PATH}/y_train.npy', y_train)
# np.save(f'{MODEL_DATA_PATH}/y_val.npy', y_val)
# np.save(f'{MODEL_DATA_PATH}/y_test.npy', y_test)

In [ ]:
# # --- SAVE COMPONENTS ---
# joblib.dump(scaler, f"{MODEL_PATH}/pca_scaler.pkl")
# joblib.dump(pca_final, f"{MODEL_PATH}/pca_final.pkl")

---  
---  